<a href="https://colab.research.google.com/github/WonJunC99/ML-in-practice/blob/master/MobileNetV3_small_TTA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏭 [2026 Edge AI Challenge] MobileNetV3-Small 제출 코드
- 모델: MobileNetV3-Small (pretrained ImageNet)
- 훈련: GPU 사용 가능
- 추론: **CPU ONLY** + Dynamic Quantization 적용

In [7]:
import os
import time
import random
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F


def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
print('✅ 시드 고정 완료')

✅ 시드 고정 완료


In [3]:
from google.colab import drive

drive.mount('/content/gdrive')

Mounted at /content/gdrive


## ⚙️ 경로 설정 (환경에 맞게 수정)

In [4]:
# ==========================================
# 🚨 본인 환경에 맞게 수정하세요!
# Colab: '/content/NEU-DET' 또는 드라이브 마운트 경로
# Kaggle: '/kaggle/input/2026-1-machine-learning-in-prace/NEU-DET_open'
# ==========================================
BASE_DATA_DIR = '/content/gdrive/MyDrive/Colab_Notebooks/MLinP_project/NEU-DET_open'
BASE_SAVE_DIR = '/content/gdrive/MyDrive/Colab_Notebooks/MLinP_project/Save'

TRAIN_DIR = os.path.join(BASE_DATA_DIR, 'train', 'images')
VAL_DIR   = os.path.join(BASE_DATA_DIR, 'validation', 'images')
TEST_DIR  = os.path.join(BASE_DATA_DIR, 'test', 'images')

TRAINED_MODELS_DIR = os.path.join(BASE_SAVE_DIR, 'trained_models')
SUBMISSION_CSV_DIR = os.path.join(BASE_SAVE_DIR, 'submission_csv')

print(f'📂 Train:  {TRAIN_DIR}')
print(f'📂 Val:    {VAL_DIR}')
print(f'📂 Test:   {TEST_DIR}')
print(f'📂 Models: {TRAINED_MODELS_DIR}')
print(f'📂 Sub:    {SUBMISSION_CSV_DIR}')

📂 Train:  /content/gdrive/MyDrive/Colab_Notebooks/MLinP_project/NEU-DET_open/train/images
📂 Val:    /content/gdrive/MyDrive/Colab_Notebooks/MLinP_project/NEU-DET_open/validation/images
📂 Test:   /content/gdrive/MyDrive/Colab_Notebooks/MLinP_project/NEU-DET_open/test/images
📂 Models: /content/gdrive/MyDrive/Colab_Notebooks/MLinP_project/Save/trained_models
📂 Sub:    /content/gdrive/MyDrive/Colab_Notebooks/MLinP_project/Save/submission_csv


## 📦 Data Loader

In [5]:
# ==========================================
# 커스텀 데이터 증강: 컨베이어 벨트 모션 블러 (도메인 특화)
# ==========================================
class RandomConveyorBeltMotionBlur(object):
    def __init__(self, kernel_size=15, p=0.3):
        self.kernel_size = kernel_size
        self.p = p

    def __call__(self, img):
        if random.random() > self.p:
            return img
        img_np = np.array(img)
        kernel = np.zeros((self.kernel_size, self.kernel_size))
        kernel[int((self.kernel_size - 1) / 2), :] = np.ones(self.kernel_size)
        kernel /= self.kernel_size
        blurred = cv2.filter2D(img_np, -1, kernel)
        return Image.fromarray(blurred)


# ==========================================
# Train Transform: 다양한 augmentation으로 일반화 향상
# ==========================================
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    RandomConveyorBeltMotionBlur(kernel_size=15, p=0.3),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


# ==========================================
# Kaggle 제출용 Test Dataset
# ==========================================
class TestDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.transform = transform
        self.img_paths = []
        self.img_names = []
        # 서브폴더(클래스별 폴더)까지 재귀 탐색
        for root, _, files in os.walk(img_dir):
            for f in sorted(files):
                if f.lower().endswith(('.jpg', '.png', '.jpeg')):
                    self.img_paths.append(os.path.join(root, f))
                    self.img_names.append(f)

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        image = Image.open(self.img_paths[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.img_names[idx]


# Data Loaders
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset   = datasets.ImageFolder(VAL_DIR,   transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

NUM_CLASSES = len(train_dataset.classes)
CLASS_NAMES = train_dataset.classes
print(f'✅ 클래스 수: {NUM_CLASSES}')
print(f'✅ 클래스 목록: {CLASS_NAMES}')
print(f'✅ Train 샘플 수: {len(train_dataset)}')
print(f'✅ Val   샘플 수: {len(val_dataset)}')

✅ 클래스 수: 6
✅ 클래스 목록: ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']
✅ Train 샘플 수: 1440
✅ Val   샘플 수: 180


## 🧠 Model: MobileNetV3-Small

In [6]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ 현재 디바이스: {DEVICE}')

# MobileNetV3-Small: Large 대비 절반 크기, CPU 추론 더 빠름
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)

# classifier[-1]: Linear(1024, 1000) → Linear(1024, NUM_CLASSES)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, NUM_CLASSES)
model = model.to(DEVICE)

# 파라미터 수 확인
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'✅ 총 파라미터 수: {total_params:.2f}M')

✅ 현재 디바이스: cpu
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 95.6MB/s]

✅ 총 파라미터 수: 1.52M


## 🚀 Training Loop

In [ ]:
from sklearn.metrics import f1_score

EPOCHS = 30

# Label Smoothing: 과적합 방지
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# AdamW + CosineAnnealingLR: 베이스라인 Adam보다 안정적
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

best_f1  = 0.0
best_acc = 0.0

for epoch in range(EPOCHS):
    # ── Train ──
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]', leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()

    # ── Validation ──
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            preds = torch.max(model(images), 1)[1].cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    val_acc = 100.0 * sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    val_f1  = f1_score(all_labels, all_preds, average='macro') * 100
    lr_now  = scheduler.get_last_lr()[0]

    print(f'Epoch [{epoch+1:2d}/{EPOCHS}] '
          f'Loss: {running_loss/len(train_loader):.4f} | '
          f'Val Acc: {val_acc:.2f}% | '
          f'Macro F1: {val_f1:.4f} | '
          f'LR: {lr_now:.6f}')

    if val_f1 > best_f1:
        best_f1  = val_f1
        best_acc = val_acc
        torch.save(model.state_dict(), os.path.join(TRAINED_MODELS_DIR, 'best_mobilev3_small.pth'))
        print(f'  💾 Best model saved! F1: {best_f1:.4f}')

print(f'\n🎉 학습 완료! Best Val Macro F1: {best_f1:.4f} / Acc: {best_acc:.2f}%')

## ⏱️ 추론 + Dynamic Quantization + submission.csv 생성
### ⚠️ 런타임을 반드시 CPU로 변경 후 실행하세요!

In [ ]:
# ==========================================
# 1. CPU 환경으로 모델 로드
# ==========================================
DEVICE = 'cpu'

model_inf = models.mobilenet_v3_small(weights=None)
model_inf.classifier[-1] = nn.Linear(model_inf.classifier[-1].in_features, NUM_CLASSES)
model_inf.load_state_dict(torch.load(os.path.join(TRAINED_MODELS_DIR, 'best_mb_epoch17_10.pth'), map_location=DEVICE))
model_inf.eval()

# ==========================================
# 2. Dynamic Quantization 적용 (CPU 추론 2~3x 가속)
# ==========================================
model_inf = torch.quantization.quantize_dynamic(
    model_inf,
    {nn.Linear},
    dtype=torch.qint8
)
print('✅ Dynamic Quantization 적용 완료')

# ==========================================
# 3. Test 데이터 로더 (batch_size 크게 → 처리량 증가)
# ==========================================
test_dataset = TestDataset(TEST_DIR, transform=test_transform)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)
print(f'✅ 테스트 이미지 수: {len(test_dataset)}')

# JIT 워밍업 (시간 측정 제외)
dummy = torch.zeros(1, 3, 224, 224)
with torch.no_grad():
    model_inf(dummy)

# ==========================================
# 4. 추론 + 시간 측정
# ==========================================
predictions = []
image_ids   = []

print('🚀 추론 시작...')

start_time = time.time()

with torch.no_grad():
    for images, img_names in tqdm(test_loader, desc='Inference'):
        preds = torch.max(model_inf(images), 1)[1]
        predictions.extend(preds.numpy())
        image_ids.extend(img_names)

end_time = time.time()
total_inference_time = end_time - start_time

print(f'\n✅ 추론 완료!')
print(f'⏱️  총 소요 시간: {total_inference_time:.4f}초')

if total_inference_time <= 1.0:
    print('🟢 1초 이내 달성! 패널티 없음')
elif total_inference_time <= 30.0:
    penalty = (total_inference_time - 1.0) * 2.5
    print(f'🟡 패널티: -{penalty:.2f}점')
else:
    print('🔴 30초 초과 — 실격 기준!')

In [ ]:
# ==========================================
# 5. submission.csv 저장
# ==========================================
submission_df = pd.DataFrame({
    'Id':                 image_ids,
    'Expected':           predictions,
    'inference_time_sec': round(total_inference_time, 4)
})

submission_path = os.path.join(SUBMISSION_CSV_DIR, 'submission.csv')
submission_df.to_csv(submission_path, index=False)

print(f'🎉 제출 파일 저장: {submission_path}')
print(f'\n예측 분포:')
print(submission_df['Expected'].value_counts().sort_index())
submission_df.head(10)